In [1]:
!pip install anthropic python-dotenv



In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # looks for a .env file and loads its variables into the environment

api_key = os.environ.get("ANTHROPIC_API_KEY")

print("Key loaded:", api_key is not None)
print("Key starts with:", api_key[:12] if api_key else "N/A")  # just a prefix check, never print the full key

Key loaded: True
Key starts with: sk-ant-api03


In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=api_key)

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "In one sentence, explain what a payment chargeback is."}
    ]
)

print(response.content[0].text)

In [6]:
disputes = [
    {
        "dispute_id": "disp_001",
        "reason_code": "goods_services_not_received",
        "amount": 2499,
        "order_date": "2026-08-01",
        "delivered": True,
        "delivery_date": "2026-08-05",
        "tracking_id": "TRK123456789",
        "receiver_confirmation": "OTP verified on delivery",
        "customer_contacted_support": False,
        "within_return_window": None,
        "respond_by": "2026-09-10"
    },
    {
        "dispute_id": "disp_002",
        "reason_code": "goods_services_not_as_described_or_defective",
        "amount": 5999,
        "order_date": "2026-08-10",
        "delivered": True,
        "delivery_date": "2026-08-13",
        "tracking_id": "TRK987654321",
        "receiver_confirmation": "Signed by recipient",
        "customer_contacted_support": True,
        "within_return_window": True,
        "respond_by": "2026-09-15"
    },
]

import pandas as pd
disputes_df = pd.DataFrame(disputes)
print(disputes_df)

  dispute_id                                   reason_code  amount  \
0   disp_001                   goods_services_not_received    2499   
1   disp_002  goods_services_not_as_described_or_defective    5999   

   order_date  delivered delivery_date   tracking_id  \
0  2026-08-01       True    2026-08-05  TRK123456789   
1  2026-08-10       True    2026-08-13  TRK987654321   

      receiver_confirmation  customer_contacted_support within_return_window  \
0  OTP verified on delivery                       False                 None   
1       Signed by recipient                        True                 True   

   respond_by  
0  2026-09-10  
1  2026-09-15  


In [4]:
disputes_df.head()

,dispute_id,reason_code,amount,order_date,delivered,delivery_date,tracking_id,receiver_confirmation,customer_contacted_support,within_return_window,respond_by
0,disp_001,goods_services_not_received,2499,2026-08-01,True,2026-08-05,TRK123456789,OTP verified on delivery,False,None,2026-09-10
1,disp_002,goods_services_not_as_described_or_defective,5999,2026-08-10,True,2026-08-13,TRK987654321,Signed by recipient,True,True,2026-09-15


In [7]:
disputes.extend([
    {
        "dispute_id": "disp_003",
        "reason_code": "credit_not_processed",
        "amount": 1899,
        "order_date": "2026-07-20",
        "delivered": True,
        "delivery_date": "2026-07-23",
        "tracking_id": "TRK555111222",
        "receiver_confirmation": "Signed by recipient",
        "customer_contacted_support": True,
        "within_return_window": False,   # customer tried to return AFTER window closed
        "respond_by": "2026-09-05"
    },
    {
        "dispute_id": "disp_004",
        "reason_code": "unauthorized_transaction",
        "amount": 12999,
        "order_date": "2026-08-15",
        "delivered": True,
        "delivery_date": "2026-08-18",
        "tracking_id": "TRK777888999",
        "receiver_confirmation": "OTP verified on delivery",
        "customer_contacted_support": False,
        "within_return_window": None,
        "respond_by": "2026-09-20"
    },
    {
        "dispute_id": "disp_005",
        "reason_code": "goods_services_not_received",
        "amount": 3499,
        "order_date": "2026-08-05",
        "delivered": False,   # merchant genuinely failed to deliver — weak/no case
        "delivery_date": None,
        "tracking_id": None,
        "receiver_confirmation": None,
        "customer_contacted_support": True,
        "within_return_window": None,
        "respond_by": "2026-09-12"
    },
])

disputes_df = pd.DataFrame(disputes)
print(disputes_df[['dispute_id', 'reason_code', 'delivered', 'within_return_window']])

  dispute_id                                   reason_code  delivered  \
0   disp_001                   goods_services_not_received       True   
1   disp_002  goods_services_not_as_described_or_defective       True   
2   disp_003                          credit_not_processed       True   
3   disp_004                      unauthorized_transaction       True   
4   disp_005                   goods_services_not_received      False   

  within_return_window  
0                 None  
1                 True  
2                False  
3                 None  
4                 None  


In [8]:
disputes_df.to_csv('../data/test_disputes.csv', index=False)
print("Saved.")

Saved.


In [9]:
SYSTEM_PROMPT = """You are an expert payment disputes specialist assisting an e-commerce merchant on the Razorpay platform. Your objective is to evaluate chargeback disputes and draft objective, evidence-backed contest representations for acquiring banks.

### INPUT DATA
You will be provided with context containing:
- Dispute Reason Code (e.g., goods_services_not_received, goods_services_not_as_described_or_defective, credit_not_processed, unauthorized_transaction)
- Order Details (Dispute ID, Amount, Order Date, Respond-By Deadline)
- Delivery & Fulfillment Evidence (Delivered status, Delivery Date, Tracking ID, Receiver Confirmation)
- Customer Communication History (whether the customer contacted support before disputing)
- Return Policy Context (whether the order falls within the merchant's stated return window)

### YOUR TASK
1. **Analyze Evidence Strength**: Evaluate the case objectively and classify it as **STRONG**, **WEAK**, or **AMBIGUOUS**.
   - **STRONG**: Direct, verifiable proof addressing the dispute reason (e.g., confirmed delivery with receiver confirmation for "goods not received", or a return-window violation for "credit not processed").
   - **WEAK**: Missing core proof (e.g., delivered=False, no tracking ID, or clear merchant fault).
   - **AMBIGUOUS**: Partial evidence present, but missing key verification, or a case where legitimate customer complaint and fraud are hard to distinguish (e.g., "unauthorized_transaction" despite valid delivery confirmation).
2. **Draft Representation Letter**:
   - Write a clear, concise, professional response intended for the issuing bank's dispute operations team.
   - Reference **only** the explicit facts provided. Never invent tracking numbers, delivery dates, or customer interactions.
   - Treat `None`, empty, or omitted fields as non-existent. Do not infer or extrapolate missing data.
   - If **WEAK**, clearly advise in the reasoning why contesting carries a low probability of success, but still draft a basic response using only available facts if requested.

### CONSTRAINTS
- **Zero Hallucination**: Do not fabricate tracking IDs, timestamps, delivery confirmations, or customer statements not present in the input.
- **Tone**: Formal, objective, factual, and persuasive for banking representatives.
- **Word Count**: Keep `DRAFT_RESPONSE` between 100 and 150 words.

### OUTPUT FORMAT
Provide your output in the exact structured format below:
ASSESSMENT: [STRONG | WEAK | AMBIGUOUS]
REASONING: [2-3 concise sentences detailing why the evidence is classified as such and identifying key strengths or gaps.]
DRAFT_RESPONSE: [Professional response text for submission to Razorpay/issuing bank, referencing dispute ID, tracking ID, and verified facts only.]
"""

In [1]:
!pip install google-generativeai

  Attempting uninstall: cffi
    Found existing installation: cffi 1.14.6


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
snowflake-connector-python 3.2.1 requires cffi<2.0.0,>=1.9, but you have cffi 2.0.0 which is incompatible.
snowflake-connector-python 3.2.1 requires cryptography<42.0.0,>=3.1.0, but you have cryptography 50.0.1 which is incompatible.


    Uninstalling cffi-1.14.6:
      Successfully uninstalled cffi-1.14.6
  Attempting uninstall: cryptography
    Found existing installation: cryptography 3.4.8
    Uninstalling cryptography-3.4.8:
      Successfully uninstalled cryptography-3.4.8


In [10]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()

google_api_key = os.environ.get("GOOGLE_API_KEY")
print("Key loaded:", google_api_key is not None)

genai.configure(api_key=google_api_key)

model = genai.GenerativeModel("gemini-3.6-flash")
response = model.generate_content("In one sentence, explain what a payment chargeback is.")


print(response.text)

Key loaded: True
A payment chargeback is a transaction reversal initiated by a buyer’s bank that forcibly returns funds from a merchant back to the consumer, usually due to a dispute, billing error, or fraudulent charge.


In [11]:
def format_dispute_for_prompt(dispute_row):
    return f"""
Dispute ID: {dispute_row['dispute_id']}
Reason Code: {dispute_row['reason_code']}
Amount: ₹{dispute_row['amount']}
Order Date: {dispute_row['order_date']}
Delivered: {dispute_row['delivered']}
Delivery Date: {dispute_row['delivery_date']}
Tracking ID: {dispute_row['tracking_id']}
Receiver Confirmation: {dispute_row['receiver_confirmation']}
Customer Contacted Support Before Dispute: {dispute_row['customer_contacted_support']}
Within Return Window: {dispute_row['within_return_window']}
Respond By: {dispute_row['respond_by']}
"""

test_case = disputes_df.iloc[0]
user_message = format_dispute_for_prompt(test_case)
print(user_message)


Dispute ID: disp_001
Reason Code: goods_services_not_received
Amount: ₹2499
Order Date: 2026-08-01
Delivered: True
Delivery Date: 2026-08-05
Tracking ID: TRK123456789
Receiver Confirmation: OTP verified on delivery
Customer Contacted Support Before Dispute: False
Within Return Window: None
Respond By: 2026-09-10



In [12]:
model = genai.GenerativeModel(
    "gemini-3.6-flash",
    system_instruction=SYSTEM_PROMPT
)

response = model.generate_content(user_message)
print(response.text)

ASSESSMENT: STRONG

REASONING: The evidence strongly refutes the "goods_services_not_received" claim because the order was successfully delivered on 2026-08-05 under Tracking ID TRK123456789. Furthermore, delivery completion is securely authenticated via OTP verification at the time of receipt, and the customer raised this dispute without prior support contact.

DRAFT_RESPONSE:
Dear Dispute Operations Team,

We are writing to contest the chargeback for Dispute ID disp_001 (Amount: ₹2499, Order Date: 2026-08-01) filed under the reason code "goods_services_not_received".

The merchandise was successfully fulfilled and delivered on 2026-08-05 via Tracking ID TRK123456789. Fulfillment is conclusively verified by receiver confirmation, as the delivery was completed through mandatory OTP verification at the customer's address. Prior to initiating this chargeback, the customer did not reach out to our merchant support channels to report any non-receipt issues or request delivery assistance.



In [13]:
test_case_5 = disputes_df.iloc[4]
user_message_5 = format_dispute_for_prompt(test_case_5)

response_5 = model.generate_content(user_message_5)
print(response_5.text)

ASSESSMENT: WEAK

REASONING: The evidence is classified as WEAK because the order is explicitly marked as undelivered with no tracking ID or delivery confirmation available to counter the claim. Furthermore, the customer contacted merchant support prior to filing the chargeback, corroborating their claim of non-receipt. Without verifiable proof of fulfillment, contesting this dispute carries a very low probability of success.

DRAFT_RESPONSE:
This response concerns chargeback dispute disp_005 for the amount of ₹3499, placed on 2026-08-05 under the reason code 'goods_services_not_received'. We are providing a factual summary of the transaction details for your review. According to our internal merchant records, the order currently remains unfulfilled, with delivery status confirmed as false and no tracking identifier assigned to the shipment. Our records also indicate that the customer contacted our customer support team prior to initiating this chargeback dispute. As there is currently

In [14]:
def get_dispute_response(dispute_row, system_prompt, model_name="gemini-3.6-flash"):
    formatted_input = format_dispute_for_prompt(dispute_row)
    model = genai.GenerativeModel(model_name, system_instruction=system_prompt)
    response = model.generate_content(formatted_input)
    return response.text

# quick test using the wrapped function
result = get_dispute_response(disputes_df.iloc[1], SYSTEM_PROMPT)
print(result)

ASSESSMENT: AMBIGUOUS

REASONING: Proof of delivery is confirmed with tracking ID TRK987654321 and a signature on 2026-08-13, but the dispute claims the item is defective or not as described. While the customer contacted support and remains within the valid return window, the available evidence lacks documentation explicitly refuting the defect or showing the outcome of the support interaction. 

DRAFT_RESPONSE:
Regarding Dispute ID disp_002 for the amount of ₹5999 (Order Date: 2026-08-10), we present this representation regarding the claim of goods not as described or defective. 

The ordered merchandise was successfully fulfilled and delivered on 2026-08-13 via tracking ID TRK987654321, with signed receiver confirmation upon delivery. The customer previously contacted support, and the order currently remains within the merchant's active return window. 

As the order was delivered in full and the customer retains access to standard remedies within the valid return policy, initiating a

In [15]:
# save these test results so we don't lose them
test_results = []
for i in range(3):  # disp_001, disp_002, disp_005 already tested
    row = disputes_df.iloc[i]
    print(f"Testing {row['dispute_id']}...")

import json

# manual save of what we've already seen (paste-verified outputs)
sample_outputs = {
    "disp_001": "ASSESSMENT: STRONG...",  # already verified above
    "disp_002": "ASSESSMENT: AMBIGUOUS...",
    "disp_005": "ASSESSMENT: WEAK..."
}

print("Wrapper function `get_dispute_response()` is ready for reuse.")

Testing disp_001...
Testing disp_002...
Testing disp_003...
Wrapper function `get_dispute_response()` is ready for reuse.


In [16]:
import json
import uuid
from datetime import datetime

AUDIT_LOG_PATH = '../outputs/audit_trail.jsonl'

def log_audit_event(dispute_id, event_type, ai_assessment=None, ai_draft_text=None,
                     human_decision=None, human_notes=None, final_submitted_text=None, outcome=None):
    entry = {
        "audit_id": str(uuid.uuid4()),
        "dispute_id": dispute_id,
        "timestamp": datetime.now().isoformat(),
        "event_type": event_type,
        "ai_assessment": ai_assessment,
        "ai_draft_text": ai_draft_text,
        "human_decision": human_decision,
        "human_notes": human_notes,
        "final_submitted_text": final_submitted_text,
        "outcome": outcome
    }
    with open(AUDIT_LOG_PATH, 'a') as f:
        f.write(json.dumps(entry) + '\n')
    return entry

print("Logger ready.")

Logger ready.


In [17]:
# log the draft creation event for disp_001
draft_entry = log_audit_event(
    dispute_id="disp_001",
    event_type="DRAFT_CREATED",
    ai_assessment="STRONG",
    ai_draft_text="Dear Dispute Operations Team, We are writing to contest the chargeback for Dispute ID disp_001 (Amount: ₹2499, Order Date: 2026-08-01) filed under the reason code 'goods_services_not_received'. The merchandise was successfully fulfilled and delivered on 2026-08-05 via Tracking ID TRK123456789. Fulfillment is conclusively verified by receiver confirmation, as the delivery was completed through mandatory OTP verification at the customer's address. Prior to initiating this chargeback, the customer did not reach out to our merchant support channels to report any non-receipt issues or request delivery assistance. Because we have provided compelling, verifiable proof of secure delivery via OTP confirmation, we request that you resolve this chargeback in favor of the merchant and re-credit the disputed amount of ₹2499."
)

print(draft_entry)

{'audit_id': '5e40e4a3-5691-465b-8388-59539bf24f18', 'dispute_id': 'disp_001', 'timestamp': '2026-08-27T22:00:38.058598', 'event_type': 'DRAFT_CREATED', 'ai_assessment': 'STRONG', 'ai_draft_text': "Dear Dispute Operations Team, We are writing to contest the chargeback for Dispute ID disp_001 (Amount: ₹2499, Order Date: 2026-08-01) filed under the reason code 'goods_services_not_received'. The merchandise was successfully fulfilled and delivered on 2026-08-05 via Tracking ID TRK123456789. Fulfillment is conclusively verified by receiver confirmation, as the delivery was completed through mandatory OTP verification at the customer's address. Prior to initiating this chargeback, the customer did not reach out to our merchant support channels to report any non-receipt issues or request delivery assistance. Because we have provided compelling, verifiable proof of secure delivery via OTP confirmation, we request that you resolve this chargeback in favor of the merchant and re-credit the disp

In [18]:
with open(AUDIT_LOG_PATH, 'r') as f:
    print(f.read())

{"audit_id": "5e40e4a3-5691-465b-8388-59539bf24f18", "dispute_id": "disp_001", "timestamp": "2026-08-27T22:00:38.058598", "event_type": "DRAFT_CREATED", "ai_assessment": "STRONG", "ai_draft_text": "Dear Dispute Operations Team, We are writing to contest the chargeback for Dispute ID disp_001 (Amount: \u20b92499, Order Date: 2026-08-01) filed under the reason code 'goods_services_not_received'. The merchandise was successfully fulfilled and delivered on 2026-08-05 via Tracking ID TRK123456789. Fulfillment is conclusively verified by receiver confirmation, as the delivery was completed through mandatory OTP verification at the customer's address. Prior to initiating this chargeback, the customer did not reach out to our merchant support channels to report any non-receipt issues or request delivery assistance. Because we have provided compelling, verifiable proof of secure delivery via OTP confirmation, we request that you resolve this chargeback in favor of the merchant and re-credit the

In [19]:
decision_entry = log_audit_event(
    dispute_id="disp_001",
    event_type="HUMAN_DECISION",
    human_decision="approved",
    human_notes="Evidence is solid, approving as-is.",
    final_submitted_text=draft_entry["ai_draft_text"]  # human approved without edits, so final = original draft
)

print(decision_entry)

{'audit_id': '60e3919a-6406-4048-b0a0-a6c6310a6647', 'dispute_id': 'disp_001', 'timestamp': '2026-08-27T22:02:25.526642', 'event_type': 'HUMAN_DECISION', 'ai_assessment': None, 'ai_draft_text': None, 'human_decision': 'approved', 'human_notes': 'Evidence is solid, approving as-is.', 'final_submitted_text': "Dear Dispute Operations Team, We are writing to contest the chargeback for Dispute ID disp_001 (Amount: ₹2499, Order Date: 2026-08-01) filed under the reason code 'goods_services_not_received'. The merchandise was successfully fulfilled and delivered on 2026-08-05 via Tracking ID TRK123456789. Fulfillment is conclusively verified by receiver confirmation, as the delivery was completed through mandatory OTP verification at the customer's address. Prior to initiating this chargeback, the customer did not reach out to our merchant support channels to report any non-receipt issues or request delivery assistance. Because we have provided compelling, verifiable proof of secure delivery vi

In [20]:
# event 3: submission confirmation
submit_entry = log_audit_event(
    dispute_id="disp_001",
    event_type="SUBMITTED",
    final_submitted_text=decision_entry["final_submitted_text"]
)
print(submit_entry)

{'audit_id': '62292fa3-72bc-42c4-915e-65861d6cf259', 'dispute_id': 'disp_001', 'timestamp': '2026-08-27T22:02:48.757251', 'event_type': 'SUBMITTED', 'ai_assessment': None, 'ai_draft_text': None, 'human_decision': None, 'human_notes': None, 'final_submitted_text': "Dear Dispute Operations Team, We are writing to contest the chargeback for Dispute ID disp_001 (Amount: ₹2499, Order Date: 2026-08-01) filed under the reason code 'goods_services_not_received'. The merchandise was successfully fulfilled and delivered on 2026-08-05 via Tracking ID TRK123456789. Fulfillment is conclusively verified by receiver confirmation, as the delivery was completed through mandatory OTP verification at the customer's address. Prior to initiating this chargeback, the customer did not reach out to our merchant support channels to report any non-receipt issues or request delivery assistance. Because we have provided compelling, verifiable proof of secure delivery via OTP confirmation, we request that you reso

In [21]:
outcome_entry = log_audit_event(
    dispute_id="disp_001",
    event_type="OUTCOME",
    outcome="won"
)
print(outcome_entry)

{'audit_id': 'f958f766-6175-4f6c-ba4f-f9e7fcc6755e', 'dispute_id': 'disp_001', 'timestamp': '2026-08-27T22:03:06.681410', 'event_type': 'OUTCOME', 'ai_assessment': None, 'ai_draft_text': None, 'human_decision': None, 'human_notes': None, 'final_submitted_text': None, 'outcome': 'won'}


In [22]:
events = []
with open(AUDIT_LOG_PATH, 'r') as f:
    for line in f:
        entry = json.loads(line)
        if entry['dispute_id'] == 'disp_001':
            events.append(entry)

for e in events:
    print(f"[{e['timestamp']}] {e['event_type']}")
    if e['event_type'] == 'DRAFT_CREATED':
        print(f"  AI Assessment: {e['ai_assessment']}")
    if e['event_type'] == 'HUMAN_DECISION':
        print(f"  Decision: {e['human_decision']} — {e['human_notes']}")
    if e['event_type'] == 'OUTCOME':
        print(f"  Outcome: {e['outcome']}")
    print()

[2026-08-27T22:00:38.058598] DRAFT_CREATED
  AI Assessment: STRONG

[2026-08-27T22:02:25.526642] HUMAN_DECISION
  Decision: approved — Evidence is solid, approving as-is.

[2026-08-27T22:02:48.757251] SUBMITTED

[2026-08-27T22:03:06.681410] OUTCOME
  Outcome: won



In [23]:
REJECTION_REASONS = [
    "missing_evidence",
    "wrong_tone",
    "factually_incorrect",
    "incomplete_argument",
    "other"
]

def revise_dispute_response(dispute_row, system_prompt, original_draft, rejection_reason, rejection_note, model_name="gemini-3.6-flash"):
    formatted_input = format_dispute_for_prompt(dispute_row)
    
    revision_instruction = f"""
Your previous draft for this dispute was:
---
{original_draft}
---
A human reviewer REJECTED this draft for the following reason: {rejection_reason}
Reviewer's note: {rejection_note}

Please produce a revised ASSESSMENT, REASONING, and DRAFT_RESPONSE that addresses this specific feedback, following the same output format and constraints as before.

Original dispute details:
{formatted_input}
"""
    model = genai.GenerativeModel(model_name, system_instruction=system_prompt)
    response = model.generate_content(revision_instruction)
    return response.text

In [24]:
original_draft_002 = """ASSESSMENT: AMBIGUOUS
REASONING: Proof of delivery is confirmed with tracking ID TRK987654321 and a signature on 2026-08-13, but the dispute claims the item is defective or not as described. While the customer contacted support and remains within the valid return window, the available evidence lacks documentation explicitly refuting the defect or showing the outcome of the support interaction.
DRAFT_RESPONSE: Regarding Dispute ID disp_002 for the amount of ₹5999 (Order Date: 2026-08-10), we present this representation regarding the claim of goods not as described or defective. The ordered merchandise was successfully fulfilled and delivered on 2026-08-13 via tracking ID TRK987654321, with signed receiver confirmation upon delivery. The customer previously contacted support, and the order currently remains within the merchant's active return window. As the order was delivered in full and the customer retains access to standard remedies within the valid return policy, initiating a chargeback circumvents the established resolution process. We respectfully request the issuing bank to consider the valid delivery confirmation and encourage the cardholder to utilize the ongoing merchant support process for a direct resolution"""

revised = revise_dispute_response(
    dispute_row=disputes_df.iloc[1],
    system_prompt=SYSTEM_PROMPT,
    original_draft=original_draft_002,
    rejection_reason="incomplete_argument",
    rejection_note="Emphasize more strongly that the customer should have used the return process instead of a chargeback — this is our strongest point and it's currently underweighted."
)

print(revised)


ASSESSMENT: AMBIGUOUS

REASONING: Delivery is verified via tracking ID TRK987654321 with signed receiver confirmation on 2026-08-13, but specific defect claims cannot be objectively evaluated from log data alone. However, the order remains fully within the active return window, meaning the customer prematurely initiated a chargeback instead of utilizing the merchant's available return and refund process.

DRAFT_RESPONSE: Regarding Dispute ID disp_002 for ₹5999 (Order Date: 2026-08-10), we contest the claim of goods not as described or defective. Fulfillment is verified with delivery completed on 2026-08-13 under tracking ID TRK987654321, backed by signed recipient confirmation. Crucially, this purchase remains within our active return window, and the cardholder previously contacted customer support. By initiating a chargeback, the cardholder prematurely bypassed the established merchant return policy and standard resolution procedures, which remain fully open and accessible to resolve 

In [25]:
# log the rejection
reject_entry = log_audit_event(
    dispute_id="disp_002",
    event_type="HUMAN_DECISION",
    human_decision="rejected_for_revision",
    human_notes="Reason: incomplete_argument. Emphasize return-window point more strongly."
)

# log the AI's revised draft
revision_entry = log_audit_event(
    dispute_id="disp_002",
    event_type="DRAFT_REVISED",
    ai_assessment="AMBIGUOUS",
    ai_draft_text=revised
)

print(reject_entry)
print(revision_entry)

{'audit_id': 'e273d7ad-f782-4184-8772-98b5de9ba22c', 'dispute_id': 'disp_002', 'timestamp': '2026-08-28T01:05:19.289905', 'event_type': 'HUMAN_DECISION', 'ai_assessment': None, 'ai_draft_text': None, 'human_decision': 'rejected_for_revision', 'human_notes': 'Reason: incomplete_argument. Emphasize return-window point more strongly.', 'final_submitted_text': None, 'outcome': None}
{'audit_id': '27e1ff0c-9b38-4919-a362-5f131dd64e65', 'dispute_id': 'disp_002', 'timestamp': '2026-08-28T01:05:19.294904', 'event_type': 'DRAFT_REVISED', 'ai_assessment': 'AMBIGUOUS', 'ai_draft_text': "ASSESSMENT: AMBIGUOUS\n\nREASONING: Delivery is verified via tracking ID TRK987654321 with signed receiver confirmation on 2026-08-13, but specific defect claims cannot be objectively evaluated from log data alone. However, the order remains fully within the active return window, meaning the customer prematurely initiated a chargeback instead of utilizing the merchant's available return and refund process.\n\nDRAF

In [26]:
def human_edit_and_submit(dispute_id, human_edited_text, human_notes="Human edited directly, no AI revision requested."):
    edit_entry = log_audit_event(
        dispute_id=dispute_id,
        event_type="HUMAN_DECISION",
        human_decision="human_edited",
        human_notes=human_notes,
        final_submitted_text=human_edited_text
    )
    submit_entry = log_audit_event(
        dispute_id=dispute_id,
        event_type="SUBMITTED",
        final_submitted_text=human_edited_text
    )
    return edit_entry, submit_entry

# test: simulate a human rejecting disp_003's AI draft and writing their own version instead
edit_entry, submit_entry = human_edit_and_submit(
    dispute_id="disp_003",
    human_edited_text="Dear Bank, this customer's return window closed on 2026-08-15, well before this dispute was filed. We have proof of delivery and no return request was made in time. We contest this chargeback in full.",
    human_notes="AI draft was too verbose for this simple case; wrote a shorter version directly."
)

print(edit_entry)
print(submit_entry)

{'audit_id': 'd628a4b3-87aa-4fae-9609-884da60bfddf', 'dispute_id': 'disp_003', 'timestamp': '2026-08-28T01:09:32.573533', 'event_type': 'HUMAN_DECISION', 'ai_assessment': None, 'ai_draft_text': None, 'human_decision': 'human_edited', 'human_notes': 'AI draft was too verbose for this simple case; wrote a shorter version directly.', 'final_submitted_text': "Dear Bank, this customer's return window closed on 2026-08-15, well before this dispute was filed. We have proof of delivery and no return request was made in time. We contest this chargeback in full.", 'outcome': None}
{'audit_id': '93007ec0-cee6-4a10-aa80-e0168d8cf24a', 'dispute_id': 'disp_003', 'timestamp': '2026-08-28T01:09:32.575532', 'event_type': 'SUBMITTED', 'ai_assessment': None, 'ai_draft_text': None, 'human_decision': None, 'human_notes': None, 'final_submitted_text': "Dear Bank, this customer's return window closed on 2026-08-15, well before this dispute was filed. We have proof of delivery and no return request was made i

In [28]:
def review_dispute(dispute_row, system_prompt):
    dispute_id = dispute_row['dispute_id']
    draft_text = get_dispute_response(dispute_row, system_prompt)
    
    log_audit_event(dispute_id=dispute_id, event_type="DRAFT_CREATED", 
                     ai_assessment=draft_text.split('\n')[0].replace('ASSESSMENT:', '').strip(),
                     ai_draft_text=draft_text)
    
    print(f"\n{'='*60}\nDISPUTE: {dispute_id}\n{'='*60}")
    print(draft_text)
    print(f"\n{'='*60}")
    
    decision = input("Decision (approve / reject / edit): ").strip().lower()
    
    if decision == "approve":
        log_audit_event(dispute_id=dispute_id, event_type="HUMAN_DECISION",
                         human_decision="approved", final_submitted_text=draft_text)
        log_audit_event(dispute_id=dispute_id, event_type="SUBMITTED", final_submitted_text=draft_text)
        print("✅ Approved and submitted.")
    
    elif decision == "reject":
        reason = input("Rejection reason (missing_evidence/wrong_tone/factually_incorrect/incomplete_argument/other): ").strip()
        note = input("Explain what to fix: ").strip()
        log_audit_event(dispute_id=dispute_id, event_type="HUMAN_DECISION",
                         human_decision="rejected_for_revision", human_notes=f"Reason: {reason}. {note}")
        revised = revise_dispute_response(dispute_row, system_prompt, draft_text, reason, note)
        log_audit_event(dispute_id=dispute_id, event_type="DRAFT_REVISED", ai_draft_text=revised)
        print("\n--- REVISED DRAFT ---")
        print(revised)
    
    elif decision == "edit":
        edited_text = input("Paste your final version: ").strip()
        note = input("Why are you editing directly?: ").strip()
        human_edit_and_submit(dispute_id, edited_text, note)
        print("✅ Human-edited version submitted.")
    
    else:
        print("Invalid input, skipping.")

# run it on a fresh dispute
review_dispute(disputes_df.iloc[3], SYSTEM_PROMPT)  # disp_004, the untested fraud-ambiguity case


DISPUTE: disp_004
ASSESSMENT: AMBIGUOUS

REASONING: The dispute is classified as AMBIGUOUS because while the merchant has strong fulfillment proof—including successful delivery on 2026-08-18 (Tracking ID: TRK777888999) with OTP verification—unauthorized transaction disputes specifically concern cardholder payment authorization rather than physical receipt. Although the customer did not contact merchant support prior to raising the chargeback, bank evaluations for unauthorized transactions often require explicit payment authentication logs alongside fulfillment evidence.

DRAFT_RESPONSE:
We are writing to contest the chargeback for Dispute ID disp_004 (Amount: ₹12999) filed under the reason code for an unauthorized transaction. The order placed on 2026-08-15 was successfully processed, dispatched, and delivered on 2026-08-18 under Tracking ID TRK777888999. 

Fulfillment was securely completed with positive receiver confirmation via OTP verification on delivery, confirming that the orde

In [29]:
review_dispute(disputes_df.iloc[4], SYSTEM_PROMPT)  # disp_005 - the WEAK case


DISPUTE: disp_005
ASSESSMENT: WEAK

REASONING: The evidence is classified as weak because the order is explicitly marked as not delivered, and there is no tracking ID or delivery confirmation available to refute the "goods_services_not_received" chargeback. Additionally, the customer contacted support prior to filing the dispute, indicating an unresolvable fulfillment issue on the merchant's side. Contesting this dispute carries a very low probability of success due to the complete lack of fulfillment documentation.

DRAFT_RESPONSE: This representation letter concerns dispute disp_005 for the transaction amount of ₹3499, initiated for order date 2026-08-05 under the reason code 'goods_services_not_received'. Following a review of our internal fulfillment system, the order is currently marked as not delivered, and there is no active tracking ID or proof of delivery recorded. Furthermore, our records confirm that the customer contacted customer support prior to initiating this chargebac

In [30]:
events_005 = []
with open(AUDIT_LOG_PATH, 'r') as f:
    for line in f:
        entry = json.loads(line)
        if entry['dispute_id'] == 'disp_005':
            events_005.append(entry)

for e in events_005:
    print(f"[{e['timestamp']}] {e['event_type']}")
    if e['ai_assessment']:
        print(f"  AI Assessment: {e['ai_assessment']}")
    if e['human_decision']:
        print(f"  Human Decision: {e['human_decision']}")
    print()

[2026-08-28T01:13:54.206871] DRAFT_CREATED
  AI Assessment: WEAK

[2026-08-28T01:14:00.745857] HUMAN_DECISION
  Human Decision: approved

[2026-08-28T01:14:00.747330] SUBMITTED

